In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

env: CUPY_ACCELERATORS=cutensor,cub
TensorLy backend: cupy


In [2]:
from moabb.datasets import *
from moabb.paradigms import P300

dataset = BNCI2014_008()
paradigm = P300(resample=32)
epochs, labels, meta = paradigm.get_data(dataset, subjects=[1], return_epochs=True)

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>



Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/paradigms/base.py:350: RuntimeWarning:

Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.



In [3]:
from mne import combine_evoked

contrast = combine_evoked([epochs['Target'].average(), epochs['NonTarget'].average()], weights=[1,-1])
contrast.plot_joint();

No projector specified for this dataset. Please consider the method self.add_proj.


In [4]:
from hoda.classification import ZScore
X = tl.tensor(epochs.get_data())
y = labels
X_st = ZScore().fit_transform(X)
X.shape

(4200, 8, 32)

In [5]:
from hoda.hoda import HODA

hoda = HODA(
        rank=None,
        max_iter=128,
        tol=1e-6,
        shrinkage='lw',
        refit_shrinkage=True,
        obj='tr',
        solver='lanczos',
        toeplitz=(1,),
        taper=False,
        extra_train_info=False,
        verbose=False,
        theta=0.2,
        forward=True       
)


In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.util import flip_signs
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage
from hoda.util import solve_gevdh

plt.style.use('default')

%load_ext line_profiler
hoda.fit_backward(X_st,y)
df = pd.DataFrame(hoda.train_info_['backward'])

In [7]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[0]))

In [8]:
px.imshow(tl.to_numpy(hoda.scatter_w_[1]))

In [9]:
0.001231992088601549
0.05504274022049101

0.05504274022049101

In [10]:
df

,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,8.056672e-01,0.000237,466517.995212
1,1,2,2,1.434084e+00,0.023565,53939.576325
2,2,1,3,6.947741e-01,0.000669,205618.909123
3,2,2,4,4.110173e-01,0.025426,38133.878978
4,3,1,5,4.709125e-01,0.000551,47959.876481
5,3,2,6,2.340618e-01,0.043656,34737.869609
6,4,1,7,4.350437e-01,0.000520,20671.057871
7,4,2,8,1.626895e-01,0.045579,35636.119676
8,5,1,9,2.058024e-01,0.000484,12677.074846
9,5,2,10,1.406826e+00,0.042560,38267.707385


In [11]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr')
    fig.show()


In [12]:
if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_rt')
    fig.show()


In [13]:

px.line(df, x='flip', y='update', log_y=True, color='mode')


In [14]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [15]:
px.line(df, x='flip', y='objective', log_y=True, color='mode')

In [16]:
from hoda.util import ridge_regression

%lprun -f ridge_regression hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

,iteration,mode,flip,update,lambda_
0,1,1,1,1.205082e+00,0.0
1,1,2,2,1.905642e+00,0.0
2,2,1,3,5.879530e-01,0.0
3,2,2,4,3.916668e-01,0.0
4,3,1,5,2.294995e-01,0.0
5,3,2,6,7.894262e-02,0.0
6,4,1,7,6.903336e-02,0.0
7,4,2,8,2.034152e-02,0.0
8,5,1,9,2.021485e-02,0.0
9,5,2,10,5.745792e-03,0.0


Timer unit: 1e-09 s

Total time: 0.0218955 s
File: /vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/util.py
Function: ridge_regression at line 116

Line #      Hits         Time  Per Hit   % Time  Line Contents
   116                                           def ridge_regression(X, Y, lambda_=0):
   117                                                   """
   118                                                   Compute the ridge regression solution for matrix Y.
   119                                                   
   120                                                   X: Input matrix (n x p)
   121                                                   Y: Target matrix (n x m)
   122                                                   lambda_: Regularization parameter
   123                                                   
   124                                                   Returns:
   125                                                   W: The ridge regression wei

In [17]:
if hoda.extra_train_info:
    px.line(df, x='flip', y='mse', log_y=True)

In [18]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [19]:
px.line(df, x='flip', y='lambda_', log_y=True, color='mode')

In [20]:
import numpy as np
from mne.viz import plot_topomap
import matplotlib.pyplot as plt
import tensorly as tl

col_wrap = 4

n_col = col_wrap
n_row = int( np.ceil(hoda.rank_[0] / col_wrap)) 
fig, axs = plt.subplots(n_row, n_col, layout='tight')
vmax = float(tl.max(tl.abs(hoda.weights_[0])))
for i in range(hoda.rank_[0]):
    w = tl.to_numpy(hoda.weights_[0][:,i])
    plot_topomap(w, epochs.info, axes=axs.flatten()[i], show=False, vlim=[-vmax, vmax])

In [21]:
from mne.viz import plot_topomap
from plotly.tools import mpl_to_plotly

# TODO spatial weights
df = pd.DataFrame(tl.to_numpy(hoda.weights_[1]))
df['time'] = epochs.times
df = df.melt(id_vars=['time'], var_name='rank', value_name='weight')
px.line(df, x='time', y='weight', facet_col='rank', facet_col_wrap=4)

In [22]:

n_row = int(np.ceil(hoda.rank_[0] / col_wrap)) 
fig, axs = plt.subplots(n_row, n_col, layout='tight')
A = hoda.aps_[0]
vmax = float(tl.max(tl.abs(A)))
for i in range(hoda.rank_[0]):
    a = tl.to_numpy(A[:,i])
    plot_topomap(a, epochs.info, axes=axs.flatten()[i], show=False, vlim=[-vmax, vmax])


In [23]:
df = pd.DataFrame(tl.to_numpy(hoda.aps_[1]))
df['time'] = epochs.times
df = df.melt(id_vars=['time'], var_name='rank', value_name='amplitude')
px.line(df, x='time', y='amplitude', facet_col='rank', facet_col_wrap=4)

In [24]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np
from sklearn.preprocessing import StandardScaler

Xt = hoda.transform(X_st)
xt = tl.to_numpy(tl.unfold(Xt,0))
xt = StandardScaler().fit_transform(xt)

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

,feature,F,p_value,significant
0,0,0.210653,6.462797e-01,False
1,1,64.766385,1.088041e-15,True
2,2,0.578975,4.467572e-01,False
3,3,794.651424,2.868869e-160,True


In [25]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [26]:
X_rec = hoda.inv_transform(Xt)
px.imshow(X_rec[5].get(), color_continuous_scale='RdBu_r', aspect='auto',     zmin=-1.5,
    zmax=1.5,)

In [27]:
from sklearn.manifold import TSNE

x_viz = TSNE(n_components=2).fit_transform(xts)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)

In [28]:
from sklearn.metrics import get_scorer

scorer = get_scorer(paradigm.scoring)
scorer([0,1],[0,1])

TypeError: _BaseScorer.__call__() missing 1 required positional argument: 'y_true'